# DistilBERT PTQ investigation on SST-2

This notebook loads the public `distil_bert_base_en_uncased` pretrained weights, fine-tunes a two-class sentiment classifier on SST-2 when no saved checkpoint exists, and compares:

1. the original FP32 classifier,
2. TensorFlow Lite built-in dynamic-range weight PTQ, and
3. the project's custom per-output-channel symmetric INT8 weight PTQ.

SST-2 is used since it is a standard single-sentence sentiment-classification task well matched to an English DistilBERT encoder. Both PTQ accuracy paths are weight-only: activations execute in FP32. Custom activation ranges are calibrated and reported, but they are not applied during inference. No fake-quantization API is used here.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import json
import sys

import keras
import keras_hub
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.lite.python import schema_py_generated as schema_fb

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.tflite_metrics import inspect_tflite
from src.models.transformer_based import (
    DISTILBERT_BASE_EN_UNCASED_PRESET,
    SST2_CLASS_NAMES,
    build_distilbert_text_classifier,
    build_distilbert_text_preprocessor,
)
from src.quantization.custom_quantization import custom_ptq, dequantize_tensor

np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"KerasHub: {keras_hub.__version__}")

TensorFlow: 2.20.0
Keras: 3.14.1
KerasHub: 0.29.1


/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

The first run downloads SST-2 and the public pretrained DistilBERT weights, fine-tunes the classifier, and saves it. Later runs load the saved `.keras` model. Set `MAX_TRAIN_SAMPLES=None` to use all 67,349 SST-2 training examples.

In [2]:
PRESET = DISTILBERT_BASE_EN_UNCASED_PRESET
SEQUENCE_LENGTH = 128
NUM_CLASSES = 2
CLASS_NAMES = SST2_CLASS_NAMES

DATA_DIR = PROJECT_ROOT / "data" / "sst2"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "distilbert_sst2_ptq"
MODEL_PATH = OUTPUT_DIR / "distilbert_sst2_fp32.keras"
BUILTIN_MODEL_PATH = OUTPUT_DIR / "distilbert_sst2_builtin_dynamic_int8.tflite"

SST2_URL = "https://dl.fbaipublicfiles.com/glue/data/SST-2.zip"
MAX_TRAIN_SAMPLES = 20_000  # Use None for the complete training split.
NUM_EVALUATION_SAMPLES = None  # SST-2 dev has 872 labeled examples.
NUM_CALIBRATION_SAMPLES = 100
TRAIN_BATCH_SIZE = 16
EVALUATION_BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 2e-5
FREEZE_BACKBONE = False
FORCE_RETRAIN = False
FORCE_RECONVERT_BUILTIN = False

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saved classifier: {MODEL_PATH}")
print(f"Built-in PTQ model: {BUILTIN_MODEL_PATH}")

Saved classifier: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/distilbert_sst2_fp32.keras
Built-in PTQ model: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/distilbert_sst2_builtin_dynamic_int8.tflite


## 2. Download and load SST-2

SST-2 contains movie-review sentences labeled as negative (`0`) or positive (`1`). The labeled `dev` split is kept for the final comparison; it is not used for weight updates.

In [3]:
train_tsv = DATA_DIR / "SST-2" / "train.tsv"
dev_tsv = DATA_DIR / "SST-2" / "dev.tsv"
if not train_tsv.exists() or not dev_tsv.exists():
    archive_path = Path(keras.utils.get_file(
        fname="SST-2.zip",
        origin=SST2_URL,
        cache_dir=str(PROJECT_ROOT),
        cache_subdir="data/sst2",
    ))
    with ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIR)

train_frame = pd.read_csv(train_tsv, sep="\t")
dev_frame = pd.read_csv(dev_tsv, sep="\t")
train_frame = train_frame.sample(frac=1.0, random_state=42).reset_index(drop=True)
if MAX_TRAIN_SAMPLES is not None:
    train_frame = train_frame.iloc[:MAX_TRAIN_SAMPLES].copy()
if NUM_EVALUATION_SAMPLES is not None:
    dev_frame = dev_frame.iloc[:NUM_EVALUATION_SAMPLES].copy()

train_texts = train_frame["sentence"].astype(str).to_numpy()
train_labels = train_frame["label"].astype(np.int32).to_numpy()
test_texts = dev_frame["sentence"].astype(str).to_numpy()
test_labels = dev_frame["label"].astype(np.int32).to_numpy()
print(f"Training examples: {len(train_texts):,}")
print(f"Evaluation examples: {len(test_texts):,}")
display(train_frame.head())

7439277/7439277 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training examples: 20,000
Evaluation examples: 872


,sentence,label
0,with outtakes in which most of the characters ...,0
1,enigma is well-made,1
2,is ) so stoked to make an important film about...,0
3,the closest thing to the experience of space t...,1
4,lose their luster,0


## 3. Tokenize text outside the quantized graph

DistilBERT receives two integer tensors: `token_ids` and `padding_mask`, each shaped `(batch, sequence_length)`. Tokenization is not a learned part of the neural network, so it remains outside both PTQ paths.

In [4]:
preprocessor = build_distilbert_text_preprocessor(
    sequence_length=SEQUENCE_LENGTH
)

def tokenize_texts(texts, batch_size=256):
    chunks = {"token_ids": [], "padding_mask": []}
    for start in range(0, len(texts), batch_size):
        encoded = preprocessor(tf.constant(texts[start : start + batch_size]))
        for name in chunks:
            chunks[name].append(encoded[name].numpy().astype(np.int32))
    return {name: np.concatenate(values) for name, values in chunks.items()}

train_inputs = tokenize_texts(train_texts)
test_inputs = tokenize_texts(test_texts)
print({name: values.shape for name, values in train_inputs.items()})

100%|██████████| 462/462 [00:00<00:00, 783kB/s]


100%|██████████| 794/794 [00:00<00:00, 476kB/s]


100%|██████████| 226k/226k [00:00<00:00, 629kB/s]


{'token_ids': (20000, 128), 'padding_mask': (20000, 128)}


## 4. Load existing weights and obtain an SST-2 classifier

If the saved SST-2 checkpoint exists, it is loaded directly. Otherwise, `build_distilbert_text_classifier()` from `src.models.transformer_based` loads the existing public DistilBERT backbone weights; only the new sentiment head starts randomly initialized. The complete classifier is then fine-tuned and cached.

In [5]:
if MODEL_PATH.exists() and not FORCE_RETRAIN:
    model = keras.models.load_model(MODEL_PATH, compile=False)
    print(f"Loaded fine-tuned weights: {MODEL_PATH}")
else:
    model = build_distilbert_text_classifier(
        num_classes=NUM_CLASSES,
        freeze_backbone=FREEZE_BACKBONE,
    )
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=LEARNING_RATE, weight_decay=0.01
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    train_dataset = (
        tf.data.Dataset.from_tensor_slices((train_inputs, train_labels))
        .shuffle(min(len(train_labels), 10_000), seed=42)
        .batch(TRAIN_BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    model.fit(train_dataset, epochs=EPOCHS)
    model.save(MODEL_PATH)
    print(f"Saved fine-tuned classifier: {MODEL_PATH}")

model.summary(expand_nested=True)

100%|██████████| 253M/253M [00:08<00:00, 32.9MB/s] 


Epoch 1/2
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3398s 3s/step - accuracy: 0.8758 - loss: 0.2928
Epoch 2/2
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 3467s 3s/step - accuracy: 0.9454 - loss: 0.1481
Saved fine-tuned classifier: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/distilbert_sst2_fp32.keras


Model: "distilbert_text_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ distil_bert_backbone          │ (None, None, 768)         │      66,362,880 │ padding_mask[0][0],        │
│ (DistilBertBackbone)          │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ token_ids (InputLayer)   │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └                          │ (None, None, 768)         │      23,834,112 │ -                          │
│ token_and_position_embedding  │                           │                 │                            │
│ (TokenAndPositionEmbedding)   │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ embeddings_layer_norm    │ (None, None, 768)         │           1,536 │ -                          │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ embeddings_dropout       │ (None, None, 768)         │               0 │ -                          │
│ (Dropout)                     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ padding_mask             │ (None, None)              │               0 │ -                          │
│ (InputLayer)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ transformer_layer_0      │ (None, None, 768)         │       7,087,872 │ -                          │
│ (TransformerEncoder)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ transformer_layer_1      │ (None, None, 768)         │       7,087,872 │ -                          │
│ (TransformerEncoder)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ transformer_layer_2      │ (None, None, 768)         │       7,087,872 │ -                          │
│ (TransformerEncoder)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│    └ transformer_layer_3      │ (None, None, 768)         │       7,087,872 │ -                          │
│ (TransformerEncoder)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 200,865,032 (766.24 MB)

 Trainable params: 66,955,010 (255.41 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 133,910,022 (510.83 MB)

## 5. Custom per-channel INT8 weight PTQ

The custom method quantizes every floating parameter tensor with rank `>= 2`, including token/position embeddings and Dense kernels. Rank-1 biases and LayerNorm parameters remain FP32. The stored INT8 arrays are dequantized once into a second Keras model so accuracy can be evaluated with FP32 operations. Representative inputs also produce activation min/max, scale, and zero-point metadata, but those activation parameters are not applied to this inference path.

In [6]:
calibration_count = min(NUM_CALIBRATION_SAMPLES, len(train_labels))
calibration_samples = [
    {name: values[index : index + 1] for name, values in train_inputs.items()}
    for index in range(calibration_count)
]
custom_results = custom_ptq(
    model,
    representative_samples=calibration_samples,
    quantize_min_rank=2,
    per_channel=True,
)
weight_result = custom_results["weights"]
activation_result = custom_results.get("activations")

custom_model = keras.models.load_model(MODEL_PATH, compile=False)
quantized_tensor_iter = iter(weight_result.tensors)
reconstructed_weights = []
for weight in model.weights:
    weight_array = weight.numpy()
    should_quantize = (
        np.issubdtype(weight_array.dtype, np.floating)
        and weight_array.ndim >= 2
    )
    if should_quantize:
        quantized_tensor = next(quantized_tensor_iter)
        reconstructed_weights.append(
            dequantize_tensor(quantized_tensor).astype(weight_array.dtype)
        )
    else:
        reconstructed_weights.append(weight_array)
if list(quantized_tensor_iter):
    raise RuntimeError("Not all custom quantized tensors were consumed.")
custom_model.set_weights(reconstructed_weights)

display(pd.DataFrame([{
    "quantized_weight_tensors": len(weight_result.tensors),
    "calibrated_activation_tensors": (
        len(activation_result.ranges) if activation_result else 0
    ),
    "fp32_parameter_mib": weight_result.fp32_size_bytes / 1024**2,
    "custom_parameter_mib": weight_result.quantized_size_bytes / 1024**2,
    "memory_reduction_percent": weight_result.memory_reduction_percent,
}]))

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 210 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


,quantized_weight_tensors,calibrated_activation_tensors,fp32_parameter_mib,custom_parameter_mib,memory_reduction_percent
0,58,4,255.413094,63.991707,74.9458


## 6. Built-in TFLite dynamic-range PTQ

Dynamic-range PTQ compresses supported constant weight tensors to INT8 while leaving activations in FP32 and keeping the integer `token_ids`/`padding_mask` inputs unchanged. This is a closer comparison to the current custom W8 evaluation than full W8A8 conversion.

In [7]:
def build_fixed_distilbert_model(classifier):
    token_ids = keras.Input(
        shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name="token_ids"
    )
    padding_mask = keras.Input(
        shape=(SEQUENCE_LENGTH,), dtype=tf.int32, name="padding_mask"
    )
    logits = classifier(
        {"token_ids": token_ids, "padding_mask": padding_mask},
        training=False,
    )
    return keras.Model(
        {"token_ids": token_ids, "padding_mask": padding_mask},
        logits,
        name="distilbert_sst2_fixed",
    )

if FORCE_RECONVERT_BUILTIN or not BUILTIN_MODEL_PATH.exists():
    fixed_model = build_fixed_distilbert_model(model)
    # TensorFlow 2.20 cannot reliably trace this KerasHub model through
    # from_keras_model(), so export a temporary SavedModel first.
    with TemporaryDirectory(dir=OUTPUT_DIR) as saved_model_dir:
        fixed_model.export(saved_model_dir, format="tf_saved_model")
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        BUILTIN_MODEL_PATH.write_bytes(converter.convert())

builtin_storage = inspect_tflite(BUILTIN_MODEL_PATH)
print(f"Built-in model: {BUILTIN_MODEL_PATH}")
print(f"Serialized size: {BUILTIN_MODEL_PATH.stat().st_size / 1024**2:.2f} MiB")
builtin_storage

INFO:tensorflow:Assets written to: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/tmp7onagkte/assets


INFO:tensorflow:Assets written to: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/tmp7onagkte/assets


Saved artifact at '/Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/tmp7onagkte'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): Dict[['token_ids', TensorSpec(shape=(None, 128), dtype=tf.int32, name='token_ids')], ['padding_mask', TensorSpec(shape=(None, 128), dtype=tf.int32, name='padding_mask')]]
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  4826181136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4826179792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4826176528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4826182864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4415342160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4826180176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4826182672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4826184208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4826177296: TensorSpe

W0000 00:00:1786654208.082764 12525768 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786654208.082885 12525768 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-13 22:50:08.085125: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/tmp7onagkte
2026-08-13 22:50:08.087226: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-13 22:50:08.087232: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/tmp7onagkte
I0000 00:00:1786654208.106254 12525768 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-13 22:50:08.109639: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-13 22:50:08.285346: I tensorflow/cc/saved_model/loader.cc

Built-in model: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/distilbert_sst2_builtin_dynamic_int8.tflite
Serialized size: 64.61 MiB


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


{'input_dtype': 'int32',
 'output_dtype': 'float32',
 'tensor_dtype_counts': {'bool': 3, 'float32': 380, 'int32': 18, 'int8': 40},
 'weight_storage_bytes': 67141780,
 'activation_tensor_storage_bytes': 121644436,
 'largest_activation_tensor_bytes': 1572864}

## 7. Tensor rank, value-count, and memory investigation

For Keras, a parameter tensor means a persistent model weight. For TFLite, graph tensors also include temporary activations, so constant buffers and all graph tensors are reported separately. Custom scale metadata and TFLite FlatBuffer metadata are excluded from raw parameter-buffer comparisons.

In [8]:
def count_tensors_by_rank(keras_model, quantize_min_rank=2):
    rows = []
    for weight in keras_model.weights:
        values = weight.numpy()
        is_quantized = (
            np.issubdtype(values.dtype, np.floating)
            and values.ndim >= quantize_min_rank
        )
        rows.append({
            "tensor": getattr(weight, "path", weight.name),
            "rank": values.ndim,
            "shape": tuple(values.shape),
            "values": values.size,
            "quantized_by_custom_ptq": is_quantized,
        })
    detail = pd.DataFrame(rows)
    summary = (
        detail.groupby("rank", as_index=False)
        .agg(
            total_tensors=("tensor", "count"),
            total_values=("values", "sum"),
            custom_quantized_tensors=("quantized_by_custom_ptq", "sum"),
        )
        .sort_values("rank")
    )
    summary["custom_unquantized_tensors"] = (
        summary["total_tensors"] - summary["custom_quantized_tensors"]
    )
    return summary, detail

rank_summary, individual_tensor_ranks = count_tensors_by_rank(model)
rank_summary

,rank,total_tensors,total_values,custom_quantized_tensors,custom_unquantized_tensors
0,1,46,48386,0,46
1,2,34,52750848,34,0
2,3,24,14155776,24,0


In [9]:
builtin_interpreter = tf.lite.Interpreter(model_path=str(BUILTIN_MODEL_PATH))
builtin_interpreter.allocate_tensors()
builtin_details = builtin_interpreter.get_tensor_details()
builtin_dtype_counts = pd.Series([
    np.dtype(detail["dtype"]).name for detail in builtin_details
]).value_counts().to_dict()

def inspect_tflite_constant_tensors(model_path, tensor_details):
    content = Path(model_path).read_bytes()
    flatbuffer_model = schema_fb.Model.GetRootAsModel(content, 0)
    subgraph = flatbuffer_model.Subgraphs(0)
    details_by_index = {int(detail["index"]): detail for detail in tensor_details}
    rows = []
    for tensor_index in range(subgraph.TensorsLength()):
        tensor = subgraph.Tensors(tensor_index)
        buffer = flatbuffer_model.Buffers(tensor.Buffer())
        if buffer.DataLength() == 0:
            continue
        detail = details_by_index.get(tensor_index)
        if detail is None:
            continue
        rows.append({
            "tensor_index": tensor_index,
            "name": detail["name"],
            "dtype": np.dtype(detail["dtype"]).name,
            "shape": tuple(int(value) for value in detail["shape"]),
            "values": int(np.prod(detail["shape"], dtype=np.int64)),
            "storage_bytes": int(buffer.DataLength()),
        })
    return pd.DataFrame(rows)

builtin_constant_tensors = inspect_tflite_constant_tensors(
    BUILTIN_MODEL_PATH, builtin_details
)
builtin_int8_constants = builtin_constant_tensors[
    builtin_constant_tensors["dtype"] == "int8"
]

original_parameter_tensors = len(model.weights)
original_parameter_values = int(sum(weight.numpy().size for weight in model.weights))
original_parameter_bytes = int(sum(weight.numpy().nbytes for weight in model.weights))
custom_quantized_values = int(sum(t.values.size for t in weight_result.tensors))
builtin_parameter_bytes = int(builtin_storage["weight_storage_bytes"])

tensor_analysis_table = pd.DataFrame([
    {
        "model": "Original FP32 DistilBERT",
        "stored_parameter_tensors": original_parameter_tensors,
        "int8_parameter_or_constant_tensors": 0,
        "total_parameter_values": original_parameter_values,
        "int8_parameter_or_constant_values": 0,
        "total_tflite_graph_tensors": None,
    },
    {
        "model": "Custom per-channel W8",
        "stored_parameter_tensors": original_parameter_tensors,
        "int8_parameter_or_constant_tensors": len(weight_result.tensors),
        "total_parameter_values": original_parameter_values,
        "int8_parameter_or_constant_values": custom_quantized_values,
        "total_tflite_graph_tensors": None,
    },
    {
        "model": "Built-in dynamic-range TFLite",
        "stored_parameter_tensors": None,
        "int8_parameter_or_constant_tensors": len(builtin_int8_constants),
        "total_parameter_values": int(builtin_constant_tensors["values"].sum()),
        "int8_parameter_or_constant_values": int(builtin_int8_constants["values"].sum()),
        "total_tflite_graph_tensors": len(builtin_details),
    },
])

memory_analysis_table = pd.DataFrame([
    {"model": "Original FP32 DistilBERT", "raw_parameter_bytes": original_parameter_bytes},
    {"model": "Custom per-channel W8", "raw_parameter_bytes": int(weight_result.quantized_size_bytes)},
    {"model": "Built-in dynamic-range TFLite", "raw_parameter_bytes": builtin_parameter_bytes},
])
memory_analysis_table["raw_parameter_mib"] = memory_analysis_table["raw_parameter_bytes"] / 1024**2
memory_analysis_table["compression_ratio_vs_fp32"] = (
    original_parameter_bytes / memory_analysis_table["raw_parameter_bytes"]
)
memory_analysis_table["memory_reduction_percent"] = (
    1 - memory_analysis_table["raw_parameter_bytes"] / original_parameter_bytes
) * 100

display(tensor_analysis_table)
display(pd.DataFrame([builtin_dtype_counts], index=["built-in graph tensor dtypes"]))
display(
    builtin_constant_tensors.groupby("dtype", as_index=False).agg(
        constant_tensors=("name", "count"),
        constant_values=("values", "sum"),
        storage_bytes=("storage_bytes", "sum"),
    )
)
display(memory_analysis_table)

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,stored_parameter_tensors,int8_parameter_or_constant_tensors,total_parameter_values,int8_parameter_or_constant_values,total_tflite_graph_tensors
0,Original FP32 DistilBERT,104.0,0,66955010,0,NaN
1,Custom per-channel W8,104.0,58,66955010,66906624,NaN
2,Built-in dynamic-range TFLite,NaN,39,66660134,66499584,441.0


,float32,int8,int32,bool
built-in graph tensor dtypes,380,40,18,3


,dtype,constant_tensors,constant_values,storage_bytes
0,float32,69,160518,642072
1,int32,12,32,128
2,int8,39,66499584,66499584


,model,raw_parameter_bytes,raw_parameter_mib,compression_ratio_vs_fp32,memory_reduction_percent
0,Original FP32 DistilBERT,267820040,255.413094,1.000000,0.000000
1,Custom per-channel W8,67100168,63.991707,3.991347,74.945800
2,Built-in dynamic-range TFLite,67141780,64.031391,3.988873,74.930263


## 8. Prediction helpers and one-example check

In [10]:
def slice_inputs(inputs, start, stop):
    return {name: values[start:stop] for name, values in inputs.items()}

def predict_keras_labels(keras_model, inputs, batch_size=EVALUATION_BATCH_SIZE):
    predictions = []
    count = len(next(iter(inputs.values())))
    for start in range(0, count, batch_size):
        logits = keras_model(
            slice_inputs(inputs, start, start + batch_size), training=False
        ).numpy()
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)

def input_value_for_detail(detail, batch):
    name = detail["name"].lower()
    if "padding_mask" in name:
        return batch["padding_mask"]
    if "token_ids" in name or "token_id" in name:
        return batch["token_ids"]
    raise KeyError(f"Cannot map TFLite input {detail['name']!r}.")

def predict_tflite_labels(model_path, inputs, batch_size=EVALUATION_BATCH_SIZE):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    predictions = []
    count = len(next(iter(inputs.values())))
    for start in range(0, count, batch_size):
        batch = slice_inputs(inputs, start, start + batch_size)
        for detail in interpreter.get_input_details():
            value = input_value_for_detail(detail, batch)
            interpreter.resize_tensor_input(detail["index"], value.shape, strict=False)
        interpreter.allocate_tensors()
        for detail in interpreter.get_input_details():
            value = input_value_for_detail(detail, batch).astype(detail["dtype"])
            interpreter.set_tensor(detail["index"], value)
        interpreter.invoke()
        logits = interpreter.get_tensor(interpreter.get_output_details()[0]["index"])
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)

sample_inputs = slice_inputs(test_inputs, 0, 1)
sample_predictions = {
    "Original FP32": int(predict_keras_labels(model, sample_inputs, 1)[0]),
    "Built-in dynamic-range PTQ": int(predict_tflite_labels(BUILTIN_MODEL_PATH, sample_inputs, 1)[0]),
    "Custom per-channel W8": int(predict_keras_labels(custom_model, sample_inputs, 1)[0]),
}
pd.DataFrame([
    {
        "model": name,
        "text": test_texts[0],
        "true_label": CLASS_NAMES[test_labels[0]],
        "prediction": CLASS_NAMES[prediction],
        "correct": prediction == test_labels[0],
    }
    for name, prediction in sample_predictions.items()
])

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,text,true_label,prediction,correct
0,Original FP32,it 's a charming and often affecting journey .,positive,positive,True
1,Built-in dynamic-range PTQ,it 's a charming and often affecting journey .,positive,positive,True
2,Custom per-channel W8,it 's a charming and often affecting journey .,positive,positive,True


## 9. Accuracy comparison

All methods use exactly the same tokenized SST-2 development examples. Because both PTQ paths are weight-only with FP32 activations, this comparison isolates weight-quantization error more fairly.

In [11]:
fp32_predictions = predict_keras_labels(model, test_inputs)
builtin_predictions = predict_tflite_labels(BUILTIN_MODEL_PATH, test_inputs)
custom_predictions = predict_keras_labels(custom_model, test_inputs)

fp32_accuracy = float(np.mean(fp32_predictions == test_labels))
builtin_accuracy = float(np.mean(builtin_predictions == test_labels))
custom_accuracy = float(np.mean(custom_predictions == test_labels))

accuracy_table = pd.DataFrame([
    {
        "model": "Original FP32 DistilBERT",
        "quantization": "FP32 baseline",
        "correct_predictions": int(np.sum(fp32_predictions == test_labels)),
        "test_examples": len(test_labels),
        "accuracy_percent": fp32_accuracy * 100,
        "drop_from_fp32_percentage_points": 0.0,
    },
    {
        "model": "Built-in dynamic-range PTQ",
        "quantization": "supported W8 constants; FP32 activations",
        "correct_predictions": int(np.sum(builtin_predictions == test_labels)),
        "test_examples": len(test_labels),
        "accuracy_percent": builtin_accuracy * 100,
        "drop_from_fp32_percentage_points": (fp32_accuracy - builtin_accuracy) * 100,
    },
    {
        "model": "Custom per-channel W8",
        "quantization": "rank >= 2 W8; FP32 Keras activations",
        "correct_predictions": int(np.sum(custom_predictions == test_labels)),
        "test_examples": len(test_labels),
        "accuracy_percent": custom_accuracy * 100,
        "drop_from_fp32_percentage_points": (fp32_accuracy - custom_accuracy) * 100,
    },
])
accuracy_table

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,quantization,correct_predictions,test_examples,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 DistilBERT,FP32 baseline,771,872,88.417431,0.000000
1,Built-in dynamic-range PTQ,supported W8 constants; FP32 activations,768,872,88.073394,0.344037
2,Custom per-channel W8,rank >= 2 W8; FP32 Keras activations,771,872,88.417431,0.000000


## 10. Save investigation results

In [12]:
accuracy_path = OUTPUT_DIR / "ptq_accuracy_comparison.csv"
rank_path = OUTPUT_DIR / "parameter_rank_summary.csv"
tensor_path = OUTPUT_DIR / "parameter_tensor_details.csv"
memory_path = OUTPUT_DIR / "parameter_memory_comparison.csv"
results_path = OUTPUT_DIR / "results.json"
accuracy_table.to_csv(accuracy_path, index=False)
rank_summary.to_csv(rank_path, index=False)
individual_tensor_ranks.to_csv(tensor_path, index=False)
memory_analysis_table.to_csv(memory_path, index=False)
results_path.write_text(json.dumps({
    "dataset": "GLUE SST-2 dev",
    "preset": PRESET,
    "sequence_length": SEQUENCE_LENGTH,
    "training_examples": int(len(train_labels)),
    "evaluation_examples": int(len(test_labels)),
    "calibration_examples": calibration_count,
    "fp32_accuracy_percent": fp32_accuracy * 100,
    "builtin_dynamic_ptq_accuracy_percent": builtin_accuracy * 100,
    "custom_weight_ptq_accuracy_percent": custom_accuracy * 100,
    "custom_activation_calibration_applied_during_inference": False,
}, indent=2) + "\n", encoding="utf-8")
print(f"Saved: {accuracy_path}")
print(f"Saved: {rank_path}")
print(f"Saved: {tensor_path}")
print(f"Saved: {memory_path}")
print(f"Saved: {results_path}")

Saved: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/ptq_accuracy_comparison.csv
Saved: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/parameter_rank_summary.csv
Saved: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/parameter_tensor_details.csv
Saved: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/parameter_memory_comparison.csv
Saved: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/distilbert_sst2_ptq/results.json


## Final DistilBERT PTQ accuracy comparison (%)

In [13]:
final_accuracy_table = accuracy_table.copy()
final_accuracy_table["accuracy_percent"] = final_accuracy_table["accuracy_percent"].map(
    lambda value: f"{value:.2f}%"
)
final_accuracy_table["drop_from_fp32_percentage_points"] = final_accuracy_table[
    "drop_from_fp32_percentage_points"
].map(lambda value: f"{value:.2f}")
final_accuracy_table

,model,quantization,correct_predictions,test_examples,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 DistilBERT,FP32 baseline,771,872,88.42%,0.00
1,Built-in dynamic-range PTQ,supported W8 constants; FP32 activations,768,872,88.07%,0.34
2,Custom per-channel W8,rank >= 2 W8; FP32 Keras activations,771,872,88.42%,0.00
